# NEOAI 2026: CuiesNLP

---

# Overview

A **stripped** transformer has reached us — its first layer, embedding table, and language modeling head have been **removed**. Despite the wreckage, the model's layers still know the answers. Your task: recover the multiple-choice predictions for **1000 questions** using whatever signal survives in this damaged network.

---

# Description

You are given a **stripped** Qwen3.5-2B language model — only layers 1–23 + final RMSNorm survived; `embed_tokens`, `layers[0]`, and `lm_head` are absent. 

Each question consists of a context, a query, and **4 answer options** (A/B/C/D), plus 2 sink options. Inputs come **pre-embedded** — for every question you receive the hidden state right after layer 0 of the original model, so you can pick up the forward pass from layer 1 without needing the embedding table.

---

# Data

All numerical data is stored in pickled lists of dictionaries; labels are CSV.

---

## model.pt + model_config/

The stripped  state_dict  and a `Qwen3_5TextConfig` to reconstruct the architecture. Load with:

```python
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM
cfg = Qwen3_5TextConfig.from_pretrained('model_config')
model = Qwen3_5ForCausalLM(cfg).cuda().to(torch.float16)
model.load_state_dict(torch.load('model.pt', weights_only=True), strict=False)
# Then replace embed_tokens / layers[0] / lm_head with no-ops
# (see baseline.ipynb for the canonical loader)
```

---

## student_train.pkl

Reference data from our world: **100 questions** with hidden labels exposed.

Each item is a dict with:

- `idx` — integer id
- `embeds_after_l0` — `np.float16` array `(seq_len, 2048)`, hidden state after layer 0 of the original full model
- `eol_positions` — `list[int]` of length `n_opts`, positions of the token of each answer option
- `correct_idx` — integer in `[0, n_opts)`, the correct option index

---

## student_val.pkl

Snapshots to predict: **1000 questions**. Same fields as train, **but `correct_idx` is removed**.

---

## sample_submission.csv

Template submission (all labels set to `0`).

Columns:

- `id` — must match `student_val.pkl[i]['idx']`
- `target` — predicted option index `∈ {0, 1, 2, 3}`

---

# Task

For every record in `student_val.pkl`, predict a single integer class label `∈ {0, 1, 2, 3}` — the index of the correct answer option among the 4 real options (sinks E/F are never correct and are not in the prediction space).

This is a **multi-class classification** problem with 4 options per question; 1000 questions total

---

# Evaluation

Submissions are evaluated using **Accuracy**:

---

# Submission File

The submission file must contain a header and follow this format:

```csv
id,target
0,2
1,3
2,0
...
```

# NEOAI 2026: CuiesNLP

---

# Обзор

До нас добрался **урезанный** трансформер — его первый слой, таблица эмбеддингов и language modeling head были **удалены**. Несмотря на разрушения, слои модели всё ещё знают ответы. Ваша задача: восстановить предсказания multiple-choice для **1000 вопросов**, используя любые сигналы, сохранившиеся в этой повреждённой сети.

---

# Описание

Вам предоставлена **урезанная** языковая модель Qwen3.5-2B — сохранились только слои 1–23 + финальный RMSNorm; `embed_tokens`, `layers[0]` и `lm_head` отсутствуют.

Каждый вопрос состоит из контекста, запроса и **4 вариантов ответа** (A/B/C/D), а также 2 sink-вариантов. Входы приходят уже **в виде эмбеддингов** — для каждого вопроса вам предоставляется hidden state сразу после layer 0 оригинальной модели, так что вы можете продолжить forward pass с layer 1 без необходимости использовать таблицу эмбеддингов.

---

# Данные

Все численные данные хранятся в pickled-списках словарей; метки находятся в CSV.

---

## model.pt + model_config/

Урезанный `state_dict` и `Qwen3_5TextConfig` для восстановления архитектуры. Загружать следующим образом:

```python
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM
cfg = Qwen3_5TextConfig.from_pretrained('model_config')
model = Qwen3_5ForCausalLM(cfg).cuda().to(torch.float16)
model.load_state_dict(torch.load('model.pt', weights_only=True), strict=False)
# Затем замените embed_tokens / layers[0] / lm_head на no-op модули
# (см. baseline.ipynb для канонического загрузчика)
```

---

## student_train.pkl

Референсные данные из нашего мира: **100 вопросов** с раскрытыми hidden labels.

Каждый элемент представляет собой словарь со следующими полями:

- `idx` — целочисленный id
- `embeds_after_l0` — `np.float16` массив `(seq_len, 2048)`, hidden state после layer 0 оригинальной полной модели
- `eol_positions` — `list[int]` длины `n_opts`, позиции токена каждого варианта ответа 
- `correct_idx` — целое число в диапазоне `[0, n_opts)`, индекс правильного варианта ответа

---

## student_val.pkl

Снимки для предсказания: **1000 вопросов**. Те же поля, что и в train, **но `correct_idx` удалён**.

---

## sample_submission.csv

Шаблон файла отправки (все метки установлены в `0`).

Столбцы:

- `id` — должен совпадать с `student_val.pkl[i]['idx']`
- `target` — предсказанный индекс варианта ответа `∈ {0, 1, 2, 3}`

---

# Задача

Для каждой записи в `student_val.pkl` предскажите единственную целочисленную метку класса `∈ {0, 1, 2, 3}` — индекс правильного варианта ответа среди 4 настоящих вариантов (sink-варианты E/F никогда не являются правильными и не входят в пространство предсказаний).

Это задача **многоклассовой классификации** с 4 вариантами ответа на вопрос; всего 1000 вопросов

---

# Оценка

Предсказания участников оцениваются с использованием **Accuracy**.

---

# Файл решения

Файл отправки должен содержать заголовок и иметь следующий формат:

```csv
id,target
0,2
1,3
2,0
...
```

In [1]:
!pip install --upgrade transformers

import os, csv, pickle, numpy as np, torch
from torch import nn
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM

HERE = '/kaggle/input/competitions/neoai-2026-day-2-cutiesnlp'
WEIGHTS = os.path.join(HERE, 'model.pt')
CONFIG  = os.path.join(HERE, 'model_config')
TRAIN   = os.path.join(HERE, 'student_train.pkl')
VAL     = os.path.join(HERE, 'student_val.pkl')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 84.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 90.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling hu

In [2]:
_train = pickle.load(open(TRAIN, 'rb'))
_val   = pickle.load(open(VAL, 'rb'))
print(f'train: {len(_train)} records   val: {len(_val)} records')
print(f'\ntrain[0] keys: {list(_train[0].keys())}')
for k, v in _train[0].items():
    print(f"  {k:<18} {type(v).__name__:<10} "
          f"shape={getattr(v, 'shape', None)}  "
          f"dtype={getattr(v, 'dtype', None)}  "
          f"value={v if not hasattr(v, 'shape') else '...'}")

for k, v in _val[0].items():
    print(f"  {k:<18} {type(v).__name__:<10} "
          f"shape={getattr(v, 'shape', None)}  "
          f"dtype={getattr(v, 'dtype', None)}  "
          f"value={v if not hasattr(v, 'shape') else '...'}")

train: 100 records   val: 1000 records

train[0] keys: ['idx', 'embeds_after_l0', 'eol_positions', 'correct_idx']
  idx                int        shape=None  dtype=None  value=0
  embeds_after_l0    ndarray    shape=(227, 2048)  dtype=float16  value=...
  eol_positions      list       shape=None  dtype=None  value=[164, 183, 199, 208]
  correct_idx        int        shape=None  dtype=None  value=0
  idx                int        shape=None  dtype=None  value=0
  embeds_after_l0    ndarray    shape=(160, 2048)  dtype=float16  value=...
  eol_positions      list       shape=None  dtype=None  value=[110, 119, 132, 141]


## Load model: config from disk + weights from .pt

In [3]:
class _Pass(nn.Module):
    def forward(self, x, *a, **k): return x

cfg = Qwen3_5TextConfig.from_pretrained(CONFIG)
model = Qwen3_5ForCausalLM(cfg).to(dtype=torch.float16, device='cuda')
model.load_state_dict(torch.load(WEIGHTS, map_location='cuda', weights_only=True), strict=False)
model.model.embed_tokens = nn.Identity()
model.model.layers[0] = _Pass()
model.lm_head = nn.Identity()
model.eval();

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


## Forward pass — collect last layer embeds at option EOL tokens

In [7]:
def collect(samples, desc):
    out = []
    for s in tqdm(samples, desc=desc):
        e = torch.from_numpy(s['embeds_after_l0']).unsqueeze(0).cuda().to(torch.float16)
        m = torch.ones(1, e.shape[1], dtype=torch.long, device='cuda')
        
        with torch.no_grad():
            outputs = model.model(
                inputs_embeds=e, 
                attention_mask=m, 
                output_hidden_states=True, 
                return_dict=True
            )
        
        h_13 = outputs.hidden_states[13][0]
        h_17 = outputs.hidden_states[17][0]
        
        eol_positions = s['eol_positions']
        h_13_opts = h_13[eol_positions].float().cpu().numpy().astype(np.float32) 
        h_17_opts = h_17[eol_positions].float().cpu().numpy().astype(np.float32)
        
        h_concat = np.concatenate([h_13_opts, h_17_opts], axis=1)
        
        out.append({
            'h': h_concat,
            'n_opts': len(eol_positions),
            'correct': s.get('correct_idx'),
        })
    return out

train = pickle.load(open(TRAIN, 'rb'))
val = pickle.load(open(VAL, 'rb'))

tr = collect(train, 'train')
va = collect(val, 'val')

val: 100%|██████████| 1000/1000 [03:08<00:00,  5.30it/s]


## Train LR + evaluate on val

In [12]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
X = np.concatenate([r['h'] for r in tr])
y = np.concatenate([[i == r['correct'] for i in range(r['n_opts'])] for r in tr]).astype(int)
sc = StandardScaler().fit(X)
clf = XGBClassifier().fit(sc.transform(X), y)
    
correct = sum(int(np.argmax(clf.predict_proba(sc.transform(r['h']))[:, 1]) == r['correct']) for r in va)
print(f'Val acc: {correct}/{len(va)} = {correct/len(va)*100:.1f}%')

Val acc: 0/1000 = 0.0%


## Generate submission CSV

In [13]:
SUB_PATH = 'submission.csv'

with open(SUB_PATH, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'target'])
    for s, r in zip(val, va):
        proba = clf.predict_proba(sc.transform(r['h']))[:, 1]
        pred = int(np.argmax(proba))
        w.writerow([s['idx'], pred])

print(f'Wrote {SUB_PATH}')
# Peek
with open(SUB_PATH) as f:
    for i, line in enumerate(f):
        if i < 5: print(' ', line.rstrip())
        else: break
print(f'  ... ({sum(1 for _ in open(SUB_PATH)) - 1} rows total)')

Wrote submission.csv
  id,target
  0,0
  1,0
  2,0
  3,2
  ... (1000 rows total)
